# Data Science Internship – February 2026
## Task 1: Build a Robust NLP Preprocessing Engine (Advanced)

This notebook implements a modular text preprocessing pipeline designed for noisy, real-world text.


## Task 1: Conceptual Understanding (Mandatory – Written)

1. **Difference between "Love" and "love" in NLP?**
   In NLP, text is usually tokenized into units (often words). Capitalization may be treated as different tokens ("Love" vs "love") unless the pipeline normalizes case. If a model is not case-normalized, it may learn separate representations for the same underlying concept.

2. **What happens if stopwords are not removed?**
   Stopwords (like *is, are, the, and*) add many high-frequency tokens that carry little semantic meaning. Keeping them can increase noise, dilute informative words, and make frequency-based features less meaningful. It can also worsen efficiency.

3. **Two scenarios where removing stopwords can be harmful**
   - **Sentiment / negation handling:** Words like *not* or phrases containing negation can change meaning. If stopwords removal is naive (e.g., removing *not*), the sentiment can flip.
   - **Search and exact phrase matching:** In queries, stopwords can be part of the user intent (e.g., "to be or not to be"). Removing stopwords can break exact matches and retrieval.

4. **Difference between stemming and lemmatization**
   - **Stemming** reduces words to a crude root using heuristic rules (e.g., *running* → *run*), but it may produce non-words.
   - **Lemmatization** reduces words to a dictionary (valid) base form using linguistic knowledge (often POS-aware), so it typically returns a real word (e.g., *better* → *good*).


In [ ]:
import re
import sys
from collections import Counter

# --- Precompiled regex patterns for performance and clarity ---

# Ensure emoji/Unicode printing works on Windows consoles (best-effort).
try:
    sys.stdout.reconfigure(encoding="utf-8")
except Exception:
    pass

# URL pattern: matches http(s) URLs and common "www" form.
URL_RE = re.compile(r"\b(?:https?://|www\.)\S+", re.IGNORECASE)

# Email-like pattern.
EMAIL_RE = re.compile(r"\b\S+@\S+\.[A-Za-z]{2,}\b")

# Collapse sequences like "soooo" -> "so" (any char repeated 3+ times becomes 1).
REPEATED_CHARS_RE = re.compile(r"(.)\1{2,}")

# Emoji range detection (used for noise scoring).
EMOJI_RE = re.compile(
    r"[\U0001F1E0-\U0001F1FF\U0001F300-\U0001F6FF\U0001F900-\U0001FAFF]+",
    flags=re.UNICODE,
)

ALLOWED_SHORT_TOKENS = {"no", "not", "i"}


In [ ]:
def preprocess_text(text):
    """Preprocess a single text string.

    Returns:
        tokens (list[str]): cleaned tokens
        cleaned_sentence (str): cleaned sentence joined by spaces
    """
    if text is None:
        return [], ""

    if not isinstance(text, str):
        text = str(text)

    text = text.strip()
    if not text:
        return [], ""

    # Convert to lowercase (required).
    text = text.lower()

    # Remove URLs and email-like patterns (required).
    text = URL_RE.sub(" ", text)
    text = EMAIL_RE.sub(" ", text)

    # Handle repeated characters like "soooo" -> "so".
    text = REPEATED_CHARS_RE.sub(r"\1", text)

    # Remove numbers (required).
    text = re.sub(r"\d+", " ", text)

    # Remove everything except letters and whitespace.
    # This also removes emojis and punctuation.
    text = re.sub(r"[^a-z\s]", " ", text)

    # Remove extra spaces (required).
    text = re.sub(r"\s+", " ", text).strip()
    if not text:
        return [], ""

    tokens = text.split(" ")

    # Remove words with length <= 2, except keep meaningful words like "no", "not".
    tokens = [
        tok for tok in tokens
        if (len(tok) > 2) or (tok in ALLOWED_SHORT_TOKENS)
    ]

    cleaned_sentence = " ".join(tokens)
    return tokens, cleaned_sentence


In [ ]:
sample_inputs = [
    "Get 100% FREE access now!!!",
    "I absolutely looooved this product 😍😍",
    "Worst service ever... 0/10",
    "Call me at 9876543210",
    "This is THE best course!!!",
    "Visit https://openai.com now!",
    "Nooooo this is baaad!!!",
    "OK OK OK I got it",
    "Win $$$ now!!! Limited offer!!!",
    "I am not happy with this",
    # Additional diversity (slang + emojis + mixed-case)
    "That movie was littt!!! 😎🔥",
    "Ngl, this update is kinda awesome tbh 😄",
]

print("---- Task 3: Stress Testing ----")

token_lists = []
clean_sentences = []

for s in sample_inputs:
    tokens, cleaned_sentence = preprocess_text(s)
    token_lists.append(tokens)
    clean_sentences.append(cleaned_sentence)

    print(f"Original Text: {s}")
    print(f"Cleaned Tokens: {tokens}")
    print(f"Cleaned Sentence: {cleaned_sentence}")
    print("-")


In [ ]:
def token_analytics_for_sentence(original_text, tokens):
    """Compute token analytics for a single sentence."""
    total_tokens = len(tokens)
    unique_tokens = len(set(tokens))
    avg_token_length = (sum(len(t) for t in tokens) / total_tokens) if total_tokens else 0.0
    return total_tokens, unique_tokens, avg_token_length


def noise_score(text):
    """Heuristic noise score (used only for the analysis question)."""
    if text is None:
        return 0

    url_count = len(URL_RE.findall(text))
    email_count = len(EMAIL_RE.findall(text))
    digit_count = len(re.findall(r"\d", text))
    emoji_count = len(EMOJI_RE.findall(text))
    repeated_count = len(REPEATED_CHARS_RE.findall(text))

    punctuation_count = len(re.findall(r"[^A-Za-z0-9\s]", text))

    return (
        url_count * 5
        + email_count * 5
        + emoji_count * 3
        + repeated_count * 2
        + digit_count * 1
        + punctuation_count * 0.2
    )


print("\n---- Task 4: Token Analytics ----")

analytics_rows = []
noise_scores = []

for original_text, tokens in zip(sample_inputs, token_lists):
    total_tokens, unique_tokens, avg_len = token_analytics_for_sentence(original_text, tokens)
    analytics_rows.append((original_text, total_tokens, unique_tokens, avg_len))
    noise_scores.append(noise_score(original_text))

    print("Original Text:", original_text)
    print("Total Tokens:", total_tokens)
    print("Unique Tokens:", unique_tokens)
    print("Average Token Length:", round(avg_len, 2))
    print("-")

most_noisy_idx = max(range(len(sample_inputs)), key=lambda i: noise_scores[i])
retained_idx = max(range(len(sample_inputs)), key=lambda i: len(token_lists[i]))

print("\nAnalysis Questions")
print("1) Which sentence had the most noise?")
print("   ->", sample_inputs[most_noisy_idx])
print("      Noise score:", round(noise_scores[most_noisy_idx], 2))

print("2) Which sentence retained the most meaningful tokens after cleaning?")
print("   ->", sample_inputs[retained_idx])
print("      Tokens:", token_lists[retained_idx])


In [ ]:
print("\n---- Task 5: Frequency Analysis ----")

all_tokens = [tok for toks in token_lists for tok in toks]
counter = Counter(all_tokens)

top_10 = counter.most_common(10)

# Least frequent words: sort by (frequency ascending, token ascending) for determinism.
least_5 = sorted(counter.items(), key=lambda kv: (kv[1], kv[0]))[:5]

print("Top 10 most frequent words:")
for word, cnt in top_10:
    print(f"{word}: {cnt}")

print("\nTop 5 least frequent words (among unique tokens):")
for word, cnt in least_5:
    print(f"{word}: {cnt}")


In [ ]:
def full_pipeline(text_list):
    """Run the full preprocessing pipeline on a list of texts.

    Expected output format:
        {
            "tokens": [...],
            "clean_sentences": [...]
        }
    """
    if text_list is None:
        return {"tokens": [], "clean_sentences": []}

    tokens_out = []
    clean_sentences_out = []

    for text in text_list:
        tokens, cleaned_sentence = preprocess_text(text)
        tokens_out.extend(tokens)
        clean_sentences_out.append(cleaned_sentence)

    return {"tokens": tokens_out, "clean_sentences": clean_sentences_out}


# Example pipeline run (uses the same stress test inputs).
pipeline_result = full_pipeline(sample_inputs)
print("\n---- Task 6: Full Pipeline Output Preview ----")
print(
    {
        "tokens_preview": pipeline_result["tokens"][:20],
        "clean_sentences_count": len(pipeline_result["clean_sentences"]),
    }
)


In [ ]:
print("\n---- Task 7: Error Handling ----")

edge_cases = [
    "",          # Empty string
    "😍😍",      # Only emojis
    "123456789",# Only numbers
]

for s in edge_cases:
    tokens, cleaned_sentence = preprocess_text(s)
    print(f"Input: {repr(s)}")
    print("Tokens:", tokens)
    print("Cleaned Sentence:", cleaned_sentence)
    print("-")
